# 第12章：回测系统

## 本章学习目标

- 理解回测引擎架构
- 掌握账户和持仓管理
- 学会配置回测参数
- 分析交易成本影响

---

## 12.1 回测系统概述

回测是量化投资的核心环节，用于验证策略在历史数据上的表现。

### 回测流程

```
┌─────────────────────────────────────────────────────────────┐
│                    回测系统架构                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐  │
│  │   策略       │ →  │   交易决策   │ →  │   执行器     │  │
│  │ (Strategy)   │    │ (Decision)   │    │ (Executor)   │  │
│  └──────────────┘    └──────────────┘    └──────────────┘  │
│         ↑                   ↓                   ↓          │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐  │
│  │   数据       │    │   交易所     │    │   账户       │  │
│  │ (Data)       │    │ (Exchange)   │    │ (Account)    │  │
│  └──────────────┘    └──────────────┘    └──────────────┘  │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 核心组件

| 组件 | 功能 |
|------|------|
| backtest() | 回测主函数 |
| Exchange | 交易所模拟，订单撮合 |
| Account | 账户管理，持仓更新 |
| Executor | 订单执行，成本计算 |

In [ ]:
import qlib
from qlib.backtest import backtest
from qlib.contrib.strategy.signal_strategy import TopkStrategy
import pandas as pd
import numpy as np

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 12.2 准备预测信号

In [ ]:
# 训练模型获取预测信号
from qlib.data.dataset import DatasetH
from qlib.contrib.data.handler import Alpha158
from qlib.contrib.model.gbdt import LGBModel

# 创建数据集
dataset = DatasetH(
    handler={
        "class": "Alpha158",
        "module_path": "qlib.contrib.data.handler",
        "kwargs": {
            "start_time": "2015-01-01",
            "end_time": "2022-12-31",
            "fit_start_time": "2015-01-01",
            "fit_end_time": "2018-12-31",
            "instruments": "csi300",
        },
    },
    segments={
        "train": ("2015-01-01", "2018-12-31"),
        "test": ("2019-01-01", "2022-12-31"),
    },
)

# 训练模型
print("训练模型...")
model = LGBModel(
    loss="mse",
    learning_rate=0.05,
    num_leaves=64,
    max_depth=6,
    n_estimators=300,
)
model.fit(dataset)
print("训练完成")

In [ ]:
# 获取预测信号
predictions = model.predict(dataset)

print(f"预测信号形状: {predictions.shape}")
predictions.head()

## 12.3 运行回测

In [ ]:
# 配置回测参数
backtest_config = {
    "strategy": {
        "class": "TopkStrategy",
        "module_path": "qlib.contrib.strategy.signal_strategy",
        "kwargs": {
            "signal": predictions,
            "topk": 50,
            "n_drop": 5,
        },
    },
    "backtest": {
        "start_time": "2020-01-01",
        "end_time": "2022-12-31",
        "account": 100000000,  # 初始资金 1亿
        "benchmark": "SH000300",  # 沪深300作为基准
        "exchange_config": {
            "limit_threshold": 0.095,  # 涨跌停阈值
            "deal_price": "vwap",  # 成交价格
        },
        "pos_type": "Position",  # 持仓类型
    },
}

In [ ]:
# 运行回测
from qlib.backtest import backtest

print("开始回测...")

# 注意：实际使用时需要正确配置
# 这里展示基本用法
try:
    report_normal, positions_normal = backtest(
        executor={
            "class": "SimulatorExecutor",
            "module_path": "qlib.backtest.executor",
            "kwargs": {
                "time_per_step": "day",
                "generate_portfolio_metrics": True,
            },
        },
        **backtest_config["backtest"],
        strategy=backtest_config["strategy"],
    )
    print("回测完成")
except Exception as e:
    print(f"回测出错: {e}")
    print("\n使用简化回测示例...")

## 12.4 回测配置详解

In [ ]:
# 回测参数详解
print("回测参数说明:")
print("=" * 60)

params = {
    "start_time": "回测开始时间",
    "end_time": "回测结束时间",
    "account": "初始资金",
    "benchmark": "基准指数",
    "exchange_config": "交易所配置",
    "pos_type": "持仓类型",
}

for param, desc in params.items():
    print(f"  {param:20s} - {desc}")

In [ ]:
# 交易所配置详解
print("\n交易所配置说明:")
print("=" * 60)

exchange_params = {
    "limit_threshold": "涨跌停阈值，超出则不交易",
    "deal_price": "成交价格: open/close/vwap",
    "open_cost": "开仓成本比例",
    "close_cost": "平仓成本比例",
    "min_cost": "最小交易成本",
    "trade_unit": "交易单位（手）",
}

for param, desc in exchange_params.items():
    print(f"  {param:20s} - {desc}")

## 12.5 交易成本分析

In [ ]:
# 交易成本组成
print("交易成本组成:")
print("=" * 60)

costs = {
    "手续费": {
        "说明": "券商收取的交易费用",
        "典型值": "0.03% (万分之三)",
    },
    "印花税": {
        "说明": "卖出时收取的税费",
        "典型值": "0.1% (千分之一)",
    },
    "滑点": {
        "说明": "实际成交价与预期价格的差异",
        "典型值": "0.1% - 0.5%",
    },
    "冲击成本": {
        "说明": "大额交易对价格的影响",
        "典型值": "取决于交易量和流动性",
    },
}

for cost, info in costs.items():
    print(f"\n{cost}:")
    print(f"  说明: {info['说明']}")
    print(f"  典型值: {info['典型值']}")

In [ ]:
# 不同成本设置对比
cost_scenarios = {
    "理想情况": {"open_cost": 0, "close_cost": 0},
    "低成本": {"open_cost": 0.0003, "close_cost": 0.0013},  # 手续费+印花税
    "中等成本": {"open_cost": 0.001, "close_cost": 0.002},
    "高成本": {"open_cost": 0.002, "close_cost": 0.003},
}

print("不同成本场景设置:")
for scenario, costs in cost_scenarios.items():
    print(f"\n{scenario}:")
    print(f"  开仓成本: {costs['open_cost']*100:.2f}%")
    print(f"  平仓成本: {costs['close_cost']*100:.2f}%")

## 12.6 账户与持仓管理

In [ ]:
# 账户对象说明
print("账户对象属性:")
print("=" * 60)

account_attrs = [
    ("cash", "可用现金"),
    ("position", "当前持仓"),
    ("portfolio_value", "组合总价值"),
    ("returns", "收益率"),
]

for attr, desc in account_attrs:
    print(f"  {attr:20s} - {desc}")

In [ ]:
# 持仓对象说明
print("\n持仓对象属性:")
print("=" * 60)

position_attrs = [
    ("stock_id", "股票代码"),
    ("amount", "持仓数量"),
    ("price", "当前价格"),
    ("weight", "持仓权重"),
    ("market_value", "市值"),
]

for attr, desc in position_attrs:
    print(f"  {attr:20s} - {desc}")

## 12.7 回测结果分析

In [ ]:
# 模拟回测结果
# 实际使用时从 backtest() 获取

import pandas as pd
import numpy as np

# 生成模拟数据
dates = pd.date_range("2020-01-01", "2022-12-31", freq="B")
np.random.seed(42)

# 模拟策略收益
strategy_returns = np.random.randn(len(dates)) * 0.01 + 0.0005
benchmark_returns = np.random.randn(len(dates)) * 0.012 + 0.0003

# 计算 cumsum
strategy_cum = (1 + pd.Series(strategy_returns, index=dates)).cumprod()
benchmark_cum = (1 + pd.Series(benchmark_returns, index=dates)).cumprod()

print("模拟回测数据生成完成")

In [ ]:
# 可视化累计收益
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(strategy_cum.index, strategy_cum.values, label='策略', linewidth=1.5)
plt.plot(benchmark_cum.index, benchmark_cum.values, label='基准', linewidth=1.5, linestyle='--')

plt.title('策略 vs 基准 累计收益')
plt.xlabel('日期')
plt.ylabel('累计收益')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 计算绩效指标
def calculate_metrics(returns, benchmark_returns=None):
    """计算绩效指标"""
    # 年化收益
    annual_return = (1 + returns.mean()) ** 252 - 1
    
    # 年化波动率
    annual_vol = returns.std() * np.sqrt(252)
    
    # 夏普比率
    sharpe = annual_return / annual_vol
    
    # 最大回撤
    cum = (1 + returns).cumprod()
    running_max = cum.cummax()
    drawdown = (cum - running_max) / running_max
    max_drawdown = drawdown.min()
    
    metrics = {
        "年化收益": f"{annual_return:.2%}",
        "年化波动率": f"{annual_vol:.2%}",
        "夏普比率": f"{sharpe:.2f}",
        "最大回撤": f"{max_drawdown:.2%}",
    }
    
    if benchmark_returns is not None:
        # 超额收益
        excess_returns = returns - benchmark_returns
        tracking_error = excess_returns.std() * np.sqrt(252)
        information_ratio = (excess_returns.mean() * 252) / tracking_error
        
        metrics["跟踪误差"] = f"{tracking_error:.2%}"
        metrics["信息比率"] = f"{information_ratio:.2f}"
    
    return metrics

# 计算指标
strategy_metrics = calculate_metrics(pd.Series(strategy_returns), pd.Series(benchmark_returns))

print("策略绩效指标:")
print("=" * 40)
for k, v in strategy_metrics.items():
    print(f"  {k}: {v}")

## 12.8 实践练习

In [ ]:
# 练习1: 配置不同成本参数运行回测
# 对比低成本和高成本场景的结果

# 你的代码



# 提示：修改 exchange_config 中的 open_cost 和 close_cost

In [ ]:
# 练习2: 分析换手率对收益的影响
# 修改 n_drop 参数，观察换手率变化

# 你的代码



# 提示：n_drop 越大，换手率越高

In [ ]:
# 练习3: 实现一个自定义执行器
# 考虑市场冲击成本

# 你的代码



# 提示：继承 SimulatorExecutor 类

## 12.9 本章小结

本章我们学习了：

1. **回测系统架构**：
   - 策略 → 交易决策 → 执行器 → 账户
   - 各组件职责

2. **回测配置**：
   - 时间范围、初始资金、基准
   - 交易所配置

3. **交易成本**：
   - 手续费、印花税、滑点、冲击成本
   - 成本对收益的影响

4. **绩效指标**：
   - 年化收益、波动率、夏普比率
   - 最大回撤、信息比率

### 关键 API 速查

```python
# 运行回测
report, positions = backtest(
    strategy={...},
    start_time="2020-01-01",
    end_time="2022-12-31",
    account=100000000,
    benchmark="SH000300",
)
```

### 下一章预告

下一章我们将学习回测报告与分析，包括：
- 生成专业回测报告
- 绩效指标分析
- 收益归因